# धडा ०५ - एजंटिक RAG


## सेटअप

हा नोटबुक Microsoft Agent Framework वापरून Agentic RAG (Retrieval-Augmented Generation) पॅटर्न प्रदर्शित करतो.

**पूर्वप्राप्ती:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — तुमचा Azure AI Search सेवा एंडपॉइंट
- `AZURE_SEARCH_API_KEY` — तुमची Azure AI Search API की
- पर्यावरण चलद्वारे कॉन्फिगर केलेले Azure OpenAI डिप्लॉयमेंट
- Azure CLI प्रमाणित (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## एजेंटिक RAG म्हणजे काय?

पारंपारिक RAG एक ठराविक प्रक्रिया अनुसरते: दस्तऐवज प्राप्त करा, नंतर प्रतिसाद तयार करा. **एजेंटिक RAG** आणखी पुढे जातो आणि एजंटला स्वायत्तता देतो की तो माहिती **कधी** आणि **कशा प्रकारे** मिळवायची ते ठरवू शकतो.

एजेंटिक RAG सह, एजंट करू शकतो:
- प्रश्नाचे उत्तर देण्यापूर्वी माहिती मिळवणे आवश्यक आहे का हे **ठरवणे**
- कोणत्या डेटा स्रोत किंवा साधनाला प्रश्न विचारायचा हे **निवडणे**
- मिळालेल्या निकालांचे **मूल्यमापन** करणे आणि जर पहिली प्रयत्न अपूर्ण असेल तर पुढील माहिती मिळवणे
- अनेक माहिती मिळविण्याच्या टप्प्यांमधील माहिती **एकत्रित करून** सुसंगत उत्तर तयार करणे

यामुळे एजंट अधिक लवचिक आणि अचूक बनतो, जे एक स्थिर retrieve-then-generate प्रक्रिया पेक्षा वेगळे आहे.


## शोध साधन तयार करणे

Agentic RAG मध्ये, बाह्य डेटा स्रोतांना **टूल्स** म्हणून वाकले जाते जे एजंट गरजेनुसार कॉल करू शकतो. यामुळे एजंटला शोध घेणे ही फक्त आणखी एक क्रिया म्हणून विचार करता येते, अनिवार्य पायरी म्हणून नाही.

खाली आम्ही एक प्रवास ज्ञानाधारित माहिती संच तयार करतो आणि एजंट कॉल करू शकेल अशा टूलच्या रूपात त्याला उपलब्ध करतो, जेथे तो गंतव्यस्थानांची माहिती शोधू शकतो.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजंट तयार करणे

आता आपण असा एजंट तयार करतो जो **उत्तर देण्यापूर्वी नेहमी माहिती मिळवण्याचे निर्देशित केलेले आहे**. एजंट त्याच्या प्रतिसादांना त्याच्या स्वतःच्या ट्रेनिंग डेटावर अवलंबून न राहता ज्ञान बेसमध्ये आधारित करण्यासाठी `search_travel_knowledge` साधन वापरतो.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## पुनरावृत्ती रिट्रीव्हल — मेकर-चेकर पॅटर्न

Agentic RAG चे एक मुख्य फायदे म्हणजे **पुनरावृत्ती रिट्रीव्हल**. एजंट त्याच्या सुरुवातीच्या निष्कर्षांची पडताळणी करण्यासाठी, सुधारण्यासाठी किंवा विस्तार करण्यासाठी अनेक राउंड सर्च करू शकतो — "मेकऱ-चेकर" वर्कफ्लो प्रमाणे:

1. **मेकऱ पायरी**: एजंट सुरुवातीची माहिती गोळा करून उत्तराचा मसुदा तयार करतो.
2. **चेकर पायरी**: एजंट तपशील पडताळण्यासाठी किंवा रिकाम्या जागा भरण्यासाठी अतिरिक्त रिट्रीव्हल करतो.

खालच्या उदाहरणात, एजंटला असे प्रश्न विचारले गेले आहेत जे अनेक गंतव्य स्थळांची तुलना करण्यासाठी आहेत, ज्यामुळे एजंटला एकापेक्षा जास्त वेळा शोध घेणे आवश्यक आहे.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

या धड्यात आपण Microsoft Agent Framework वापरून **Agentic RAG** प्रणाली कशी तयार करायची हे शिकलात:

- **Agentic RAG** एजंटला माहिती कधी मिळवायची हे स्वायत्तपणे ठरवण्याची सोय देते, ज्यामुळे माहिती मिळवणे निश्चित नसून गतिशील होते.
- **टूल्स डेटास्रोत म्हणून**: बाह्य ज्ञान आधार (जसे की Azure AI Search) हे टूल्स म्हणून गुंडाळले जातात जे एजंट वापरू शकतो.
- **पुनरावृत्ती माहिती मिळवणे**: मेकर-चेकर पद्धत एजंटला अनेक शोध चक्र करायला सक्षम करते — शोधणे, पडताळणी करणे, आणि सुधारणा करणे — अंतिम उत्तर निर्माण करण्याआधी.

उत्पादनात, आपण in-memory `TRAVEL_KNOWLEDGE_BASE` ऐवजी मोठ्या प्रमाणावर प्रवास दस्तऐवज मिळवण्यासाठी प्रत्यक्ष Azure AI Search निर्देशांक वापराल.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
हा दस्तऐवज AI भाषांतर सेव्हिस [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून भाषांतरित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, परंतु कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये चुका किंवा अचूकतेचे तुटेपण असू शकते. मूळ दस्तऐवज त्याच्या स्थानिक भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाच्या माहिती साठी व्यावसायिक मानवी भाषांतर सल्लागार करणे शिफारस केले आहे. या भाषांतराच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थलागीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
